# TissueAgent cell-annotation benchmark

This demo covers developing human heart, B-cell lymphoma (BCL), and Han mouse brain Stereo-seq, with TissueAgent and four baselines: GPTCellType, CellTypist, Biomni, and SpatialAgent. It prepares a selection-blind benchmark and runs the selected methods. Biomni and SpatialAgent use separate upstream environments. See `docs/cell_annotation_agent_baselines.md` for setup and for redrawing the committed full-cohort comparison without rerunning any methods.

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'demo':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

DATASET = 'developing_human_heart'  # developing_human_heart | bcl | han_mouse_brain_stereoseq
RUN_MODE = 'quick'                  # quick | full
METHODS = ['tissueagent', 'celltypist', 'gptcelltype', 'biomni', 'spatialagent']
BASELINE_MODELS = {
    'gptcelltype': 'gpt-5.1',
    'biomni': 'gpt-5.1',
    'spatialagent': 'gpt-5.1',
}
print({'dataset': DATASET, 'run_mode': RUN_MODE, 'methods': METHODS})

## Prepare and validate

Preparation reuses checksum-verified local files. BCL and Han Stereo-seq inputs must first be built with their dataset preparation modules; see the setup guide. It downloads only a missing reference declared by the selected manifest. Ground truth is kept separate from the agent input.

In [ ]:
from demo.cell_annotation.benchmarks import prepare_benchmark

prepared = prepare_benchmark(DATASET, run_mode=RUN_MODE)
prepared

## Run TissueAgent

The API key must already be inherited by this kernel. The notebook never displays or saves it.

In [ ]:
if 'tissueagent' in METHODS:
    if not os.environ.get('OPENAI_API_KEY'):
        raise RuntimeError('OPENAI_API_KEY is not visible; restart Jupyter from the shell where it is exported.')
    from demo.cell_annotation.tissueagent_runner import run_tissueagent
    tissueagent_result = run_tissueagent(prepared)
    display(tissueagent_result)

## Run baselines directly

CellTypist, GPTCellType, Biomni, and SpatialAgent are independent benchmark methods, not TissueAgent tools. Biomni and SpatialAgent use the external Python environments configured by `BIOMNI_PYTHON`, `SPATIALAGENT_PYTHON`, and `SPATIALAGENT_REPO`; see `docs/cell_annotation_agent_baselines.md`. Change `BASELINE_MODELS` above to compare another LLM without changing runner code.

In [ ]:
from demo.cell_annotation.baselines import (
    run_biomni,
    run_celltypist,
    run_gptcelltype,
    run_spatialagent,
)

baseline_results = {}
if 'celltypist' in METHODS:
    baseline_results['celltypist'] = run_celltypist(prepared)
if 'gptcelltype' in METHODS:
    baseline_results['gptcelltype'] = run_gptcelltype(
        prepared, model=BASELINE_MODELS['gptcelltype']
    )
if 'biomni' in METHODS:
    baseline_results['biomni'] = run_biomni(
        prepared, model=BASELINE_MODELS['biomni']
    )
if 'spatialagent' in METHODS:
    baseline_results['spatialagent'] = run_spatialagent(
        prepared, model=BASELINE_MODELS['spatialagent']
    )
baseline_results

## Evaluate

This is a direct smoke-test evaluation: unmapped, failed, or missing predictions are scored as `Unassigned`. Metrics and label audits are written to the run directory. It does not reproduce the published full-cohort protocol, which also uses a frozen truth-blind name mapping, excludes BCL's B14 doublets, and scores Han Stereo-seq in the shared Cell Ontology label space. Use the matched-task comparison driver described in `docs/cell_annotation_agent_baselines.md` for that protocol, or redraw the saved comparison tables without running inference.

In [ ]:
from demo.cell_annotation.evaluation import evaluate_predictions

metrics = evaluate_predictions(prepared)
metrics